# PANDA — train one model family on all 5 folds

This notebook trains one configurable model family across all folds, then
runs `src.oof` to compute per-fold QWK plus the 5-fold mean. Publish the
resulting weights as a Kaggle Dataset, then use `02c_ensemble_oof.ipynb` to
measure a fair Week 2 ensemble.

**Inputs:** `panda-resized-train-data-512x512` (xhlulu)
**Settings:** GPU T4 ON, Internet ON


In [ ]:
REPO = 'https://github.com/Shashaboii/AIMI_Panda_Challenge.git'
BRANCH = 'trackC'  # trackC contains the current Kaggle notebook + OOF updates
FOLDS = [0, 1, 2, 3, 4]
EPOCHS = 6
BACKBONE = 'efficientnet-b0'
LOSS = 'smoothl1'   # smoothl1, mse, ordinal
DROPOUT = 0.3
FEATURE_TAG = None
ORDINAL_MODE = 'threshold'  # threshold or expected when LOSS='ordinal'


In [ ]:
import os
import subprocess

if os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, '/kaggle/working/repo'], check=True)
subprocess.run(['git', '-C', '/kaggle/working/repo', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
!pip install -q efficientnet_pytorch


In [ ]:
import glob

candidates = (
    glob.glob('/kaggle/input/*resized*512*/train_images/train_images')
    + glob.glob('/kaggle/input/*resized*512*/train_images')
    + glob.glob('/kaggle/input/**/train_images', recursive=True)
)
IMAGE_DIR = next((c for c in candidates if os.path.isdir(c) and len(os.listdir(c)) > 5000), None)
if IMAGE_DIR is None:
    raise RuntimeError(f'Could not find image dir. Input contents: {os.listdir("/kaggle/input")}')
print('IMAGE_DIR:', IMAGE_DIR, 'files:', len(os.listdir(IMAGE_DIR)))


In [ ]:
import os
import subprocess
import sys
import time

sys.path.insert(0, '/kaggle/working/repo')

def build_run_tag():
    parts = [BACKBONE.replace('-', '')]
    if FEATURE_TAG:
        parts.append(FEATURE_TAG)
    if LOSS != 'smoothl1':
        parts.append(LOSS)
    return '_'.join(parts)

def build_weight_name(fold):
    return f'{build_run_tag()}_fold{fold}.pth'

WEIGHT_PATTERN = build_weight_name('{fold}')
OOF_CSV = f'/kaggle/working/{build_run_tag()}_oof_predictions.csv'
print('Weight pattern:', WEIGHT_PATTERN)
print('OOF output:', OOF_CSV)


In [ ]:
for fold in FOLDS:
    weight_path = f'/kaggle/working/{build_weight_name(fold)}'
    if os.path.exists(weight_path):
        print(f'fold {fold}: weights already exist, skipping')
        continue
    print(f'\n=== Training fold {fold} ===')
    cmd = [
        'python', '-m', 'src.train',
        '--fold', str(fold),
        '--folds-csv', '/kaggle/working/repo/data/train_folds.csv',
        '--image-dir', IMAGE_DIR,
        '--epochs', str(EPOCHS),
        '--backbone', BACKBONE,
        '--loss', LOSS,
        '--dropout', str(DROPOUT),
        '--output-dir', '/kaggle/working',
    ]
    if FEATURE_TAG is not None:
        cmd += ['--feature-tag', FEATURE_TAG]
    t0 = time.time()
    result = subprocess.run(cmd, cwd='/kaggle/working/repo')
    print(f'fold {fold} finished in {(time.time() - t0) / 60:.1f} min, exit code {result.returncode}')
    if result.returncode != 0:
        raise RuntimeError(f'Training fold {fold} failed')


In [ ]:
result = subprocess.run(
    [
        'python', '-m', 'src.oof',
        '--folds-csv', '/kaggle/working/repo/data/train_folds.csv',
        '--image-dir', IMAGE_DIR,
        '--weights-dir', '/kaggle/working',
        '--weight-pattern', WEIGHT_PATTERN,
        '--output-csv', OOF_CSV,
        '--backbone', BACKBONE,
        '--ordinal-mode', ORDINAL_MODE,
    ],
    cwd='/kaggle/working/repo',
)
if result.returncode != 0:
    raise RuntimeError('OOF evaluation failed')


In [ ]:
import glob
import os

import pandas as pd
from src.eval import confusion_matrix_str, qwk

resolved_oof_csv = OOF_CSV
if not os.path.exists(resolved_oof_csv):
    candidates = sorted(glob.glob('/kaggle/working/*_oof_predictions.csv'))
    if len(candidates) == 1:
        resolved_oof_csv = candidates[0]
        print(f'WARN: expected {OOF_CSV} but found {resolved_oof_csv}; using that file instead')
    else:
        raise FileNotFoundError(
            f'OOF CSV not found at {OOF_CSV}. Available OOF CSVs: {candidates if candidates else "none"}'
        )

oof = pd.read_csv(resolved_oof_csv)
print('OOF CSV:', resolved_oof_csv)
print(f'OOF rows: {len(oof)}')
print(f'Global OOF QWK: {qwk(oof.pred_raw.values, oof.isup_grade.values):.4f}')
print()
print('Confusion matrix (rows = true, cols = predicted):')
print(confusion_matrix_str(oof.pred_raw.values, oof.isup_grade.values))


In [ ]:
for f in sorted(os.listdir('/kaggle/working')):
    p = f'/kaggle/working/{f}'
    if os.path.isfile(p):
        print(f'{f}  ({os.path.getsize(p) / 1e6:.2f} MB)')
